# UERANSIM UE Bandwidth & Latency Analysis

Data collected from the `uesimtun0` tunnel interface of the UERANSIM UE connected to Open5GS 5G core network.

- Traffic was generated via variable-size ICMP pings through the 5G data plane (UE → UPF)
- Metrics: cumulative RX/TX bytes, instantaneous throughput (kbps), and RTT (ms)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Raw data: time(s), cumulative_rx_bytes, cumulative_tx_bytes, rtt_ms
raw = [
    (1,  0,      592,    0.706),
    (2,  5364,   5956,   0.424),
    (3,  5448,   6040,   0.353),
    (4,  12672,  13364,  0.410),
    (5,  18036,  18728,  0.275),
    (6,  18120,  18812,  0.479),
    (7,  25344,  26136,  0.306),
    (8,  30708,  31500,  0.419),
    (9,  30792,  31584,  0.332),
    (10, 38016,  38908,  0.351),
    (11, 43380,  44272,  0.497),
    (12, 43464,  44356,  0.330),
    (13, 50688,  51680,  0.277),
    (14, 56052,  57044,  0.310),
    (15, 56136,  57128,  0.357),
    (16, 63360,  64452,  0.295),
    (17, 68724,  69816,  0.286),
    (18, 68808,  69900,  0.353),
    (19, 76032,  77224,  0.462),
    (20, 81396,  82636,  0.394),
    (21, 81480,  82720,  0.305),
    (22, 88704,  90044,  0.312),
    (23, 94068,  95408,  0.340),
    (24, 94152,  95492,  0.462),
    (25, 101376, 102816, 0.277),
    (26, 106740, 108180, 0.298),
    (27, 106824, 108264, 0.358),
    (28, 114048, 115588, 0.323),
    (29, 119412, 120952, 0.309),
    (30, 119496, 121036, 0.376),
    (31, 126720, 128360, 0.282),
    (32, 132084, 133724, 0.276),
    (33, 132168, 133808, 0.385),
    (34, 139392, 141132, 0.461),
    (35, 144756, 146496, 0.302),
    (36, 144840, 146580, 0.370),
    (37, 152064, 153904, 0.306),
    (38, 157428, 159268, 0.254),
    (39, 157512, 159352, 0.364),
    (40, 164736, 166676, 0.379),
    (41, 170100, 172040, 0.442),
    (42, 170184, 172124, 0.347),
]

t    = np.array([r[0] for r in raw])
rx_b = np.array([r[1] for r in raw])
tx_b = np.array([r[2] for r in raw])
rtt  = np.array([r[3] for r in raw])

# Compute instantaneous throughput (kbps) between samples
rx_kbps = np.diff(rx_b) * 8 / 1000  # bytes -> kbits, per 1s interval
tx_kbps = np.diff(tx_b) * 8 / 1000
t_diff  = t[1:]  # time axis for diff'd values

print(f"Total RX: {rx_b[-1]/1024:.1f} KB | Total TX: {tx_b[-1]/1024:.1f} KB")
print(f"Avg RX throughput: {rx_kbps.mean():.1f} kbps | Avg TX throughput: {tx_kbps.mean():.1f} kbps")
print(f"Avg RTT: {rtt.mean():.3f} ms | Min RTT: {rtt.min():.3f} ms | Max RTT: {rtt.max():.3f} ms")

In [ ]:
fig = plt.figure(figsize=(14, 10))
fig.suptitle('UERANSIM UE – 5G Data Plane Performance (uesimtun0 → UPF)', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# --- Plot 1: Instantaneous RX throughput ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.fill_between(t_diff, rx_kbps, alpha=0.4, color='steelblue')
ax1.plot(t_diff, rx_kbps, color='steelblue', linewidth=1.5, label='RX')
ax1.axhline(rx_kbps.mean(), color='steelblue', linestyle='--', linewidth=1, alpha=0.7, label=f'Mean {rx_kbps.mean():.0f} kbps')
ax1.set_title('Downlink (RX) Throughput')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Throughput (kbps)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Instantaneous TX throughput ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.fill_between(t_diff, tx_kbps, alpha=0.4, color='tomato')
ax2.plot(t_diff, tx_kbps, color='tomato', linewidth=1.5, label='TX')
ax2.axhline(tx_kbps.mean(), color='tomato', linestyle='--', linewidth=1, alpha=0.7, label=f'Mean {tx_kbps.mean():.0f} kbps')
ax2.set_title('Uplink (TX) Throughput')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Throughput (kbps)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# --- Plot 3: Combined RX+TX overlay ---
ax3 = fig.add_subplot(gs[1, :])
ax3.plot(t_diff, rx_kbps, color='steelblue', linewidth=2, label='RX (Downlink)')
ax3.plot(t_diff, tx_kbps, color='tomato', linewidth=2, linestyle='--', label='TX (Uplink)')
ax3.set_title('RX vs TX Throughput Over Time')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Throughput (kbps)')
ax3.legend()
ax3.grid(True, alpha=0.3)
# shade high-traffic bursts (every 3s pattern from ping traffic)
for x in range(3, int(t_diff[-1])+1, 3):
    ax3.axvspan(x-0.5, x+1.5, alpha=0.07, color='green', label='_nolegend_')

# --- Plot 4: RTT ---
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(t, rtt, color='mediumpurple', linewidth=1.5, marker='o', markersize=3)
ax4.fill_between(t, rtt, alpha=0.2, color='mediumpurple')
ax4.axhline(rtt.mean(), color='mediumpurple', linestyle='--', linewidth=1, label=f'Mean {rtt.mean():.3f} ms')
ax4.set_title('Round-Trip Time (RTT)')
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('RTT (ms)')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# --- Plot 5: Cumulative bytes ---
ax5 = fig.add_subplot(gs[2, 1])
ax5.plot(t, rx_b / 1024, color='steelblue', linewidth=2, label='RX (KB)')
ax5.plot(t, tx_b / 1024, color='tomato', linewidth=2, linestyle='--', label='TX (KB)')
ax5.set_title('Cumulative Bytes Transferred')
ax5.set_xlabel('Time (s)')
ax5.set_ylabel('Data (KB)')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

plt.savefig('/home/w5/ors/ORANSlice/ue_bandwidth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to ue_bandwidth.png')

In [ ]:
# Summary statistics
print('=== UE Traffic Summary ===')
print(f'Duration:           {t[-1]}s')
print(f'Total data RX:      {rx_b[-1]/1024:.1f} KB ({rx_b[-1]/1024/1024*1000:.1f} MB/s avg)')
print(f'Total data TX:      {tx_b[-1]/1024:.1f} KB')
print(f'Peak RX throughput: {rx_kbps.max():.1f} kbps at t={t_diff[rx_kbps.argmax()]}s')
print(f'Peak TX throughput: {tx_kbps.max():.1f} kbps at t={t_diff[tx_kbps.argmax()]}s')
print(f'Avg RX throughput:  {rx_kbps.mean():.1f} kbps')
print(f'Avg TX throughput:  {tx_kbps.mean():.1f} kbps')
print(f'RTT mean:           {rtt.mean():.3f} ms')
print(f'RTT min/max:        {rtt.min():.3f} / {rtt.max():.3f} ms')
print(f'RTT std dev:        {rtt.std():.3f} ms')